# 2. Databricks: Ossie to Metric View and Back (S3 Backbone)

This notebook reads the Ossie file from S3, builds a Unity Catalog Metric View,
queries it, adds a measure, and exports Ossie back to S3 for the return trip.

The underlying data tables are Snowflake-managed Iceberg tables on the same S3
bucket -- no CSV upload or table creation needed.

Prerequisites:
- External location `ossie-interop-s3` exists pointing to `s3://YOUR-BUCKET-NAME/`
- Storage credential `ossie-s3-credential` is configured
- Notebook 1 has been run (Ossie file exists on S3)

## Step 1 - Install the Apache Ossie Databricks converter

In [0]:
%pip install "git+https://github.com/apache/ossie.git@01058aa416423cf43a74e7f9fb7f5f70981a418e#subdirectory=converters/databricks"

In [0]:
dbutils.library.restartPython()

## Step 2 - Configuration

Names match the Snowflake database/schema so Ossie table references work on both sides.

In [0]:
CATALOG = "demos"

# The metric view lives in this flow's own schema, so the demo, live and
# snowflake_managed flows can all be set up at once without colliding.
SCHEMA      = "demo_semantic_interop"
METRIC_VIEW = f"{CATALOG}.{SCHEMA}.sales_metric_view"

# The Iceberg tables are shared by all three flows and stay where Snowflake wrote
# them. They cannot be duplicated per flow: Snowflake appends a random suffix to each
# base location, and the discovery below scans iceberg/ for those suffixes, so a second
# copy would match twice and keep whichever the listing happened to return last.
DATA_SCHEMA = "ext_semantic_interop"

# Namespaces drive the Ossie `source:` rewrite, so they point at the DATA schema.
SF_NAMESPACE  = f"DEMOS.{DATA_SCHEMA.upper()}"
DBX_NAMESPACE = f"{CATALOG}.{DATA_SCHEMA}"

S3_BUCKET = "s3://snowflake-ossie-interop"  # <-- Replace with your S3 bucket

# Each flow reads and writes its own S3 prefix, so a live sync running in the
# background cannot overwrite the file this demo just produced.
OSSIE_PREFIX   = f"{S3_BUCKET}/ossie/demo"
OSSIE_FROM_SF  = f"{OSSIE_PREFIX}/ossie_from_snowflake.yaml"
OSSIE_FROM_DBX = f"{OSSIE_PREFIX}/ossie_from_databricks.yaml"

print(f"Metric view: {METRIC_VIEW}")
print(f"Reading Ossie from: {OSSIE_FROM_SF}")

## Step 3 - Register Iceberg tables from S3

Snowflake-managed Iceberg tables store data as Parquet with standard Iceberg metadata on S3.
Snowflake appends a random suffix to the base location (e.g. `customers.6EWgT0se/`), so we
discover the actual paths dynamically and register them as Iceberg tables in Unity Catalog.

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{DATA_SCHEMA}")

# Discover Snowflake-managed Iceberg table paths on S3.
# Snowflake appends a random suffix to the base_location (e.g. customers.6EWgT0se/).
# We find the actual paths and register them as Iceberg tables in Unity Catalog.
import re

files = dbutils.fs.ls(f"{S3_BUCKET}/iceberg/")
table_paths = {}
for f in files:
    match = re.match(r"(customers|orders)\.\w+/$", f.name)
    if match:
        table_paths[match.group(1)] = f.path.rstrip("/")

print("Discovered Iceberg table paths:", table_paths)

for table_name, s3_path in table_paths.items():
    spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{DATA_SCHEMA}.{table_name}")
    spark.sql(f"""
        CREATE OR REPLACE TABLE {CATALOG}.{DATA_SCHEMA}.{table_name}
        AS SELECT * FROM read_files('{s3_path}/data', format => 'parquet')
    """)
    print(f"Registered {CATALOG}.{DATA_SCHEMA}.{table_name} -> {s3_path}")

In [0]:
# Verify data
display(spark.sql(f"""
  SELECT region, SUM(order_amount) total_amount, COUNT(order_id) order_count, SUM(order_qty) total_qty
  FROM {CATALOG}.{DATA_SCHEMA}.orders JOIN {CATALOG}.{DATA_SCHEMA}.customers USING (customer_id)
  GROUP BY region ORDER BY region"""))
# expect EAST 750/5/12, WEST 700/5/11

## Step 4 - The spec-version and dialect shim

Bridges Snowflake's Ossie 0.1.1 format and the Apache converter's 0.2.0.dev0 format.

In [0]:
import json
import re
import yaml

CONVERTER_OSSIE_VERSION = "0.2.0.dev0"
SNOWFLAKE_OSSIE_VERSION = "0.1.1"
SNOWFLAKE_DIALECT = "SNOWFLAKE"
ANSI_DIALECT = "ANSI_SQL"
DATABRICKS_DIALECT = "DATABRICKS"


def _relabel_dialects(expression_obj, frm, to):
    if not isinstance(expression_obj, dict):
        return
    for d in expression_obj.get("dialects", []) or []:
        if d.get("dialect") == frm:
            d["dialect"] = to


def snowflake_to_converter(ossie_yaml, drop_fact_fields=True):
    root = yaml.safe_load(ossie_yaml)
    root["version"] = CONVERTER_OSSIE_VERSION
    for model in root.get("semantic_model", []) or []:
        hoisted = []
        for ds in model.get("datasets", []) or []:
            ds_name = ds.get("name", "")
            qual = re.compile(re.escape(ds_name) + r"\.", re.IGNORECASE)
            kept_ext = []
            for ext in ds.get("custom_extensions", []) or []:
                if ext.get("vendor_name") == SNOWFLAKE_DIALECT:
                    blob = json.loads(ext.get("data") or "{}")
                    for m in blob.get("metrics", []) or []:
                        expr = qual.sub("", m["expr"])
                        metric = {"name": m["name"], "expression": {"dialects": [{"dialect": ANSI_DIALECT, "expression": expr}]}}
                        # Carry the business metadata, not just the maths. Snowflake keeps
                        # synonyms in this vendor blob; the converter reads them from
                        # object-form ai_context and writes them to the Metric View's
                        # `synonyms` field, where a Databricks user can see and edit them.
                        if m.get("description"):
                            metric["description"] = m["description"]
                        if m.get("synonyms"):
                            metric["ai_context"] = {"synonyms": list(m["synonyms"])}
                        hoisted.append(metric)
                else:
                    kept_ext.append(ext)
            if kept_ext:
                ds["custom_extensions"] = kept_ext
            else:
                ds.pop("custom_extensions", None)
            new_fields = []
            for f in ds.get("fields", []) or []:
                _relabel_dialects(f.get("expression"), SNOWFLAKE_DIALECT, ANSI_DIALECT)
                for ext in f.pop("custom_extensions", []) or []:
                    if ext.get("vendor_name") != SNOWFLAKE_DIALECT:
                        continue
                    try:
                        syn = (json.loads(ext.get("data") or "{}") or {}).get("synonyms")
                    except (ValueError, TypeError):
                        continue
                    if syn:                      # same trick for dimension synonyms
                        f["ai_context"] = {"synonyms": list(syn)}
                if drop_fact_fields and "dimension" not in f:
                    continue
                new_fields.append(f)
            if new_fields:
                ds["fields"] = new_fields
            else:
                ds.pop("fields", None)
        if hoisted:
            model["metrics"] = (model.get("metrics", []) or []) + hoisted

        # Metrics that are ALREADY model-level need the same treatment as hoisted ones.
        # This is the second-hop case: converter_to_snowflake leaves metrics at model
        # level, labelled SNOWFLAKE and re-qualified as SUM(ORDERS.order_amount). Without
        # this the converter finds "no DATABRICKS/ANSI_SQL dialect" and silently drops
        # every metric, so a second round trip loses the whole measure set.
        ds_names = [ds.get("name", "") for ds in model.get("datasets", []) or [] if ds.get("name")]
        for m in model.get("metrics", []) or []:
            _relabel_dialects(m.get("expression"), SNOWFLAKE_DIALECT, ANSI_DIALECT)
            for d in (m.get("expression") or {}).get("dialects", []) or []:
                if "expression" not in d:
                    continue
                for ds_name in ds_names:
                    d["expression"] = re.sub(
                        re.escape(ds_name) + r"\.", "", d["expression"], flags=re.IGNORECASE)

        # Drop primary_key / unique_keys before the converter sees them.
        #
        # A Metric View has nowhere to put a primary key. The converter uses one only to
        # set `rely.at_most_one_match` on a matching join, and warns loudly that it is
        # doing so. Since we strip `rely` anyway, the key contributes nothing here except
        # two UserWarnings on every import, which look like errors during a demo.
        #
        # Nothing is lost: converter_to_snowflake rebuilds the key from the join columns.
        for ds in model.get("datasets", []) or []:
            ds.pop("primary_key", None)
            ds.pop("unique_keys", None)
    # Break shared object references to prevent YAML anchor/alias syntax (&id001/*id001)
    # which Snowflake's parser cannot handle.
    root = json.loads(json.dumps(root))
    return yaml.safe_dump(root, sort_keys=False)


def _pop_synonyms(obj):
    """Take synonyms off an object, from object-form ai_context or a bare key."""
    ctx = obj.get("ai_context")
    if isinstance(ctx, dict):
        syn = ctx.get("synonyms")
        del obj["ai_context"]
        if syn:
            return list(syn)
    return list(obj.pop("synonyms", None) or [])


def _keep_synonyms_as_extension(obj):
    """Move synonyms into the SNOWFLAKE vendor extension, the form Snowflake emits.

    Snowflake will not accept object-form ai_context at all: the import fails with
    "Cannot deserialize value of type `java.lang.String` from Object value". The Databricks
    converter produces exactly that for anything carrying synonyms.

    Verified against SYSTEM$CREATE_SEMANTIC_VIEW_FROM_OSSIE_YAML with a DDL-created view as
    a control: Snowflake does NOT currently ingest synonyms from Ossie in any form. They are
    written into the extension anyway, because the Ossie file is the shared artifact and they
    survive there for other consumers and for a future Snowflake release.

    Do not try to carry metric synonyms by moving metrics into the dataset blob: that form is
    opaque to the importer and the metrics disappear entirely.
    """
    syn = _pop_synonyms(obj)
    if syn:
        obj.setdefault("custom_extensions", []).append(
            {"vendor_name": SNOWFLAKE_DIALECT, "data": json.dumps({"synonyms": syn})})


def _drop_object_ai_context(node):
    """Remove any object-form `ai_context` left after synonyms have been moved."""
    if isinstance(node, dict):
        if isinstance(node.get("ai_context"), dict):
            del node["ai_context"]
        for value in node.values():
            _drop_object_ai_context(value)
    elif isinstance(node, list):
        for item in node:
            _drop_object_ai_context(item)


def converter_to_snowflake(ossie_yaml, dialect=SNOWFLAKE_DIALECT, model_name=None):
    root = yaml.safe_load(ossie_yaml)
    root["version"] = SNOWFLAKE_OSSIE_VERSION
    for model in root.get("semantic_model", []) or []:
        datasets = model.get("datasets", []) or []
        name_map = {}
        for ds in datasets:
            old_name = ds["name"]
            ds["name"] = old_name.upper()
            if old_name != ds["name"]:
                name_map[old_name] = ds["name"]
        for rel in model.get("relationships", []) or []:
            if "from" in rel:
                rel["from"] = rel["from"].upper()
            if "to" in rel:
                rel["to"] = rel["to"].upper()
        # Restore primary_key on datasets (Snowflake requires it for relationship validation).
        # First: convert unique_keys back to primary_key if present.
        for ds in datasets:
            if "unique_keys" in ds and ds["unique_keys"]:
                ds["primary_key"] = ds["unique_keys"][0]
                del ds["unique_keys"]
        # Rebuild the primary key on the referenced side of each relationship.
        #
        # A Metric View expresses a join but has no concept of a primary key, so the key is
        # gone by the time the model comes back. Snowflake will not accept the relationship
        # without it:
        #
        #     The referenced key in the relationship 'ORDERS REFERENCES CUSTOMERS' must be
        #     the primary or unique key of the referenced entity.
        #
        # The join's `to_columns` are exactly that key. The column is NOT added as a field:
        # Snowflake's own export lists primary_key on a dataset whose fields do not include
        # it, and adding it would make it a queryable dimension, which it is not.
        by_name = {ds.get("name"): ds for ds in datasets}
        for rel in model.get("relationships", []) or []:
            target = by_name.get(rel.get("to"))
            keys = [str(c).upper() for c in rel.get("to_columns") or []]
            if target is not None and keys and not target.get("primary_key"):
                target["primary_key"] = keys

        fact_ds_name = datasets[0]["name"] if datasets else None
        for ds in datasets:
            is_fact = ds.get("name") == fact_ds_name
            for f in ds.get("fields", []) or []:
                _relabel_dialects(f.get("expression"), DATABRICKS_DIALECT, dialect)
                if not is_fact:
                    f.setdefault("dimension", {})
        fact_cols = []
        ref_re = re.compile(re.escape(fact_ds_name) + r"\.([A-Za-z_]\w*)") if fact_ds_name else None
        for m in model.get("metrics", []) or []:
            _relabel_dialects(m.get("expression"), DATABRICKS_DIALECT, dialect)
            if not fact_ds_name:
                continue
            for d in (m.get("expression") or {}).get("dialects", []) or []:
                if "expression" in d:
                    for old, new in name_map.items():
                        d["expression"] = re.sub(r"\b" + re.escape(old) + r"\.", new + ".", d["expression"])
                    d["expression"] = _qualify_columns(d["expression"], fact_ds_name)
                    for c in ref_re.findall(d["expression"]):
                        if c not in fact_cols:
                            fact_cols.append(c)
        if fact_ds_name and fact_cols:
            fact_ds = datasets[0]
            existing = {f["name"].lower() for f in fact_ds.get("fields", []) or []}
            flds = fact_ds.setdefault("fields", [])
            for c in fact_cols:
                if c.lower() not in existing:
                    flds.append({"name": c.upper(), "expression": {"dialects": [{"dialect": dialect, "expression": c}]}})
        for ds in datasets:
            for f in ds.get("fields", []) or []:
                _keep_synonyms_as_extension(f)
        for m in model.get("metrics", []) or []:
            _keep_synonyms_as_extension(m)

        if model_name:
            model["name"] = model_name

    # Last, so it cannot delete synonyms before they have been moved.
    _drop_object_ai_context(root)
    # Break shared object references to prevent YAML anchor/alias syntax (&id001/*id001)
    # which Snowflake's parser cannot handle.
    root = json.loads(json.dumps(root))
    return yaml.safe_dump(root, sort_keys=False)


def _qualify_columns(expr, table):
    return re.sub(r"(?<![\w.])([A-Za-z_]\w*)(?!\s*\()(?![\w.])", lambda m: f"{table}.{m.group(1)}", expr)

## Step 5 - Read Ossie from S3 and convert to Metric View

In [0]:
from ossie_databricks import convert_ossie_to_metric_view, convert_metric_view_to_ossie

ossie_v1 = dbutils.fs.head(OSSIE_FROM_SF)

# Guard against backslash doubling from cross-platform transfer
try:
    yaml.safe_load(ossie_v1)
except yaml.YAMLError:
    ossie_v1 = ossie_v1.replace('\\\\', '\\\\'[0:1])
    yaml.safe_load(ossie_v1)

print(ossie_v1[:500])

In [0]:
converter_ready = snowflake_to_converter(ossie_v1)
mv_yaml = convert_ossie_to_metric_view(converter_ready)
mv_yaml = mv_yaml.replace(SF_NAMESPACE, DBX_NAMESPACE)

# Strip unsupported fields (rely) for older Databricks serdes
UNSUPPORTED_JOIN_FIELDS = ("rely",)
def strip_unsupported_fields(mv_yaml_text):
    mv = yaml.safe_load(mv_yaml_text)
    def clean(joins):
        for j in joins or []:
            for field in UNSUPPORTED_JOIN_FIELDS:
                j.pop(field, None)
            clean(j.get("joins"))
    clean(mv.get("joins"))
    return yaml.safe_dump(mv, sort_keys=False)

mv_yaml = strip_unsupported_fields(mv_yaml)
print(mv_yaml)

## Step 6 - Create and query the Metric View

In [0]:
def create_metric_view(fqname, yaml_body):
    spark.sql('CREATE OR REPLACE VIEW ' + fqname + ' WITH METRICS LANGUAGE YAML AS $$\n' + yaml_body + '\n$$')

create_metric_view(METRIC_VIEW, mv_yaml)
print('created', METRIC_VIEW)

In [0]:
display(spark.sql(f"""
  SELECT region,
         MEASURE(order_count) AS order_count,
         MEASURE(total_order_amount) AS total_order_amount
  FROM {METRIC_VIEW}
  GROUP BY region ORDER BY region
"""))

## Step 7 - Add a new measure: TOTAL_QUANTITY

This measure is added on the Databricks side and will travel back to Snowflake.

In [0]:
mv = yaml.safe_load(mv_yaml)
mv.setdefault('measures', []).append({'name': 'TOTAL_QUANTITY', 'expr': 'SUM(order_qty)'})
create_metric_view(METRIC_VIEW, yaml.safe_dump(mv, sort_keys=False))
print('added TOTAL_QUANTITY')

In [0]:
display(spark.sql(f"""
  SELECT region,
         MEASURE(total_quantity) AS total_quantity,
         MEASURE(total_order_amount) AS total_order_amount,
         MEASURE(order_count) AS order_count
  FROM {METRIC_VIEW}
  GROUP BY region ORDER BY region
"""))
# expect EAST 12/750/5, WEST 11/700/5

## Step 8 - Export updated Ossie back to S3

Read the deployed Metric View YAML, convert back to Snowflake-importable Ossie,
and write directly to S3.

In [0]:
def get_metric_view_yaml(metric_view_name):
    ddl = spark.sql(f'SHOW CREATE TABLE {metric_view_name}').collect()[0][0]
    start = ddl.index('$') + 2
    end = ddl.index('$', start)
    return ddl[start:end].strip()

mv_yaml_v2 = get_metric_view_yaml(METRIC_VIEW)
ossie_out = convert_metric_view_to_ossie(mv_yaml_v2)
ossie_out = ossie_out.replace(DBX_NAMESPACE, SF_NAMESPACE)
ossie_v2 = converter_to_snowflake(ossie_out, model_name='SALES_SV_V2')

dbutils.fs.put(OSSIE_FROM_DBX, ossie_v2, overwrite=True)
print(ossie_v2)
print(f'\nWritten to {OSSIE_FROM_DBX}')

## Step 9 - Sync Function (for live demo)

To run as a scheduled job: create a Databricks Job with this notebook, schedule every
1 minute. Pause/resume from the Jobs UI during the demo.

In [0]:
def sync_from_s3():
    """Read latest Ossie from Snowflake, update Metric View, export back."""
    ossie_in = dbutils.fs.head(OSSIE_FROM_SF)
    try:
        yaml.safe_load(ossie_in)
    except yaml.YAMLError:
        ossie_in = ossie_in.replace('\\\\', '\\\\'[0:1])

    cr = snowflake_to_converter(ossie_in)
    mv_body = convert_ossie_to_metric_view(cr)
    mv_body = mv_body.replace(SF_NAMESPACE, DBX_NAMESPACE)
    mv_body = strip_unsupported_fields(mv_body)
    create_metric_view(METRIC_VIEW, mv_body)

    mv_current = get_metric_view_yaml(METRIC_VIEW)
    ossie_out = convert_metric_view_to_ossie(mv_current)
    ossie_out = ossie_out.replace(DBX_NAMESPACE, SF_NAMESPACE)
    ossie_back = converter_to_snowflake(ossie_out, model_name='SALES_SV_V2')
    dbutils.fs.put(OSSIE_FROM_DBX, ossie_back, overwrite=True)
    import datetime
    print(f'Synced at {datetime.datetime.now()}')

# Uncomment to run once:
# sync_from_s3()

## Done

The updated Ossie file (with `TOTAL_QUANTITY`) is at
`s3://YOUR-BUCKET-NAME/ossie/demo/ossie_from_databricks.yaml`.
Open notebook 3 in Snowflake to import it.